# ACC 380K — A random forest that reads handwriting

The eight-student tree was small enough to grow by hand. This notebook uses the same tools on 42,000 handwritten digits. You will watch one tree overfit, then see a forest do better, and measure how the number of trees and the amount of training data change the result.

**Notebook version 0.1.0 · November 11, 2026**

Run the cells from top to bottom. In Colab, choose **File → Save a copy in Drive** if you want your own copy. There is no API key and no file to upload. One cell downloads the digits file. A full run takes a few minutes on a free runtime.

## What you should be able to do

- See an image as one row of 784 numbers.
- Split data into training rows and held-out rows.
- Show that a deeper tree keeps improving on training digits after it stops improving on held-out digits.
- Fit a `RandomForestClassifier` and read its error rate.
- Explain the out-of-bag score.

## Setup

`random_state` is fixed, so a second run gives the same split, the same forests, and the same numbers.

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_URL = "https://acc380k-data.nicholashallman.workers.dev/public/rf/hand_writing.csv"
RANDOM_STATE = 38026

## Load the digits

Each row is one handwritten digit. `label` is the digit a person says it is. The other 784 columns are the pixels of a 28 × 28 image, from 0 (black) to 255 (white).

In [ ]:
# The data server turns away requests that do not say who is asking, so we give a name.
digits = pd.read_csv(DATA_URL, storage_options={"User-Agent": "acc380k-notebook"})
print("Rows:", f"{len(digits):,}", "| columns:", digits.shape[1])
digits.head()

## One digit as numbers

Take the 784 pixel values of one row and lay them out 28 to a line. A dot stands for 0. This grid is all the forest ever sees.

In [ ]:
# Try it: change ROW to look at a different digit.
ROW = 1

pixels = digits.drop(columns="label")
image = pixels.iloc[ROW].to_numpy().reshape(28, 28)

print("Label:", digits["label"].iloc[ROW])
for line in image:
    print(" ".join(f"{value:3d}" if value > 0 else "  ." for value in line))

## The same numbers as images

`imshow` colors each number: 0 is black, 255 is white.

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for row, axis in enumerate(axes.ravel()):
    axis.imshow(pixels.iloc[row].to_numpy().reshape(28, 28), cmap="gray")
    axis.set_title("label: " + str(digits["label"].iloc[row]))
    axis.axis("off")
plt.show()

## Hold out some digits

A model can look good on rows it has already seen. To get honest evidence, set aside 20% of the digits. The forest trains on the rest and never sees the held-out rows until we score it.

In [ ]:
X = digits.drop(columns="label")
y = digits["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("Training digits:", f"{len(X_train):,}")
print("Held-out digits:", f"{len(X_test):,}")

## One tree, deeper and deeper

`max_depth` is how many questions deep a tree may grow. This cell grows a tree at each depth and scores it twice: on the digits it trained on, and on the held-out digits. It also scores a 40-tree forest at the same depth.

To keep this fast, it trains on the first 6,000 training digits.

In [ ]:
X_small = X_train.iloc[:6000]
y_small = y_train.iloc[:6000]

rows = []
for depth in [1, 2, 4, 6, 8, 10, 12, 15, 20]:
    one_tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    one_tree.fit(X_small, y_small)
    forest = RandomForestClassifier(n_estimators=40, max_depth=depth, random_state=RANDOM_STATE, n_jobs=-1)
    forest.fit(X_small, y_small)
    rows.append({
        "max_depth": depth,
        "tree_on_training": one_tree.score(X_small, y_small),
        "tree_on_held_out": one_tree.score(X_test, y_test),
        "forest_on_held_out": forest.score(X_test, y_test),
    })

depth_table = pd.DataFrame(rows).set_index("max_depth")
depth_table.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depth_table["tree_on_training"], marker="o", label="One tree, training digits")
plt.plot(depth_table["tree_on_held_out"], marker="o", label="One tree, held-out digits")
plt.plot(depth_table["forest_on_held_out"], marker="o", label="40-tree forest, held-out digits")
plt.xlabel("max_depth (questions deep)")
plt.ylabel("Share of digits right")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Read the curve

On its own training digits, the tree keeps climbing until it gets nearly every one right. On held-out digits it stops improving at about ten questions deep. Past that point the extra questions memorize quirks of the training digits. That gap is **overfitting**.

The forest averages 40 trees that each overfit in a different way. Their mistakes partly cancel, so the forest ends well above the single tree.

## Fit a forest on all the training digits

The rest of the notebook reports the **error rate**: the share of held-out digits the forest gets wrong. Lower is better.

In [ ]:
forest = RandomForestClassifier(n_estimators=25, random_state=RANDOM_STATE, n_jobs=-1)

started = time.time()
forest.fit(X_train, y_train)
print("Fit seconds:", round(time.time() - started))

error_rate = 1 - forest.score(X_test, y_test)
print(f"Held-out error rate with 25 trees: {error_rate:.1%}")

## More trees

How many trees are enough? Start with one tree and keep adding. `warm_start=True` tells the forest to keep the trees it already has and only grow the new ones.

In [ ]:
forest = RandomForestClassifier(n_estimators=1, warm_start=True, random_state=RANDOM_STATE, n_jobs=-1)

error_by_trees = {}
for n_trees in [1, 5, 10, 20, 40, 80, 160]:
    forest.set_params(n_estimators=n_trees)
    forest.fit(X_train, y_train)
    error_by_trees[n_trees] = 1 - forest.score(X_test, y_test)
    print(f"{n_trees:>3} trees: held-out error rate {error_by_trees[n_trees]:.1%}")

plt.figure(figsize=(8, 5))
plt.plot(pd.Series(error_by_trees), marker="o")
plt.xlabel("Number of trees")
plt.ylabel("Held-out error rate")
plt.grid(alpha=0.3)
plt.show()

## More training digits

Now hold the forest at 40 trees and change how many digits it trains on.

In [ ]:
error_by_rows = {}
for n_rows in [500, 1000, 2000, 5000, 10000, 20000, len(X_train)]:
    forest = RandomForestClassifier(n_estimators=40, random_state=RANDOM_STATE, n_jobs=-1)
    forest.fit(X_train.iloc[:n_rows], y_train.iloc[:n_rows])
    error_by_rows[n_rows] = 1 - forest.score(X_test, y_test)
    print(f"{n_rows:>6,} training digits: held-out error rate {error_by_rows[n_rows]:.1%}")

plt.figure(figsize=(8, 5))
plt.plot(pd.Series(error_by_rows), marker="o")
plt.xlabel("Number of training digits")
plt.ylabel("Held-out error rate")
plt.grid(alpha=0.3)
plt.show()

## Read the two curves

Both curves drop fast and then flatten. Going from 1 tree to 20 removes most of the error; going from 80 to 160 removes very little. More training digits keep helping for longer, but each doubling buys less than the one before.

## The out-of-bag score

Each tree trains on a bootstrap sample: rows drawn with replacement. About a third of the training rows are left out of any one tree's sample. Those rows are **out of bag** for that tree.

With `oob_score=True`, the forest scores every training row using only the trees that never saw it. That gives a held-out check without setting any rows aside. Compare it with the score on our real held-out digits.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1)
forest.fit(X_train, y_train)

print(f"Out-of-bag score:  {forest.oob_score_:.1%}")
print(f"Held-out score:    {forest.score(X_test, y_test):.1%}")

## Look at the mistakes

The forest's guesses for the first ten held-out digits, then ten digits it got wrong. Would you have read them correctly?

In [ ]:
guesses = pd.Series(forest.predict(X_test), index=y_test.index, name="forest_guess")
comparison = pd.DataFrame({"true_label": y_test, "forest_guess": guesses})
print(comparison.head(10).to_string())

wrong = comparison[comparison["true_label"] != comparison["forest_guess"]]
print()
print("Wrong:", len(wrong), "of", f"{len(comparison):,}", "held-out digits")

figure, axes = plt.subplots(2, 5, figsize=(10, 4.8))
for (row_label, row), axis in zip(wrong.head(10).iterrows(), axes.ravel()):
    axis.imshow(X_test.loc[row_label].to_numpy().reshape(28, 28), cmap="gray")
    axis.set_title(f"true {row['true_label']}, guessed {row['forest_guess']}")
    axis.axis("off")
plt.show()

## Try it

1. **Look at another digit.** In the "One digit as numbers" cell, change `ROW` and rerun. Can you read the digit from the numbers alone?
2. **Let every split see every pixel.** In the out-of-bag cell, add `max_features=None` to the forest and rerun. It takes much longer. Is the score better or worse than before? Why might hiding most pixels at each split help?
3. **Use shallow trees.** In the same cell, replace `max_features=None` with `max_depth=5`. How much does the held-out score fall? Find that depth on the overfitting curve.